# 125. SlimOT on Tiny ImageNet, blur 0.1 and Peer Transport 0.5, three seeds

Two branches, one per card, over more than one session - see below for
how to carry it forward. Nothing in this notebook needs editing: run it as it is.

The blur at 0.1 with Peer Transport at 0.5 was the best branch on
held-out train images (JM 58.49 against JE 57.79, one seed). Here it
runs on the full train split and is scored on the official val split,
at the three seeds of EQ, FP and FR; notebooks 125 and 126 together
give all three. Notebooks 123 and 124 run the blur alone at the same
seeds, so the two can be compared seed by seed.

| | |
|---|---|
| JT | EQ at seed 1995, blur 0.1 and Peer Transport 0.5 |
| JU | FP at seed 2026, blur 0.1 and Peer Transport 0.5 |

**How to read it.** Each branch against SlimOT at the same seed (the
"SlimOT" column), then the mean over the three seeds against SlimOT
58.99, AlphaNet 59.05 and US-Net 58.58. Across seeds SlimOT moves by
about 0.5 here.

## Data

Attach a Tiny ImageNet dataset (the one notebook 57 used), and CIFAR-100
as well; the notebook's data cell expects it.

## Timing

About 525 s an epoch for the method on a T4, so about 14.5 hours.
Plan on two sessions. When the first ends, Save Version, attach that
version's output (pick the version number, not Latest) and run again:
the notebook finds the checkpoints.

## `jt_k_heads4_r18_tiny_eps010_peer_half`

See apps/cifar100_jt_k_heads4_r18_tiny_eps010_peer_half.yml for every setting and why.

## `ju_k_heads4_r18_tiny_eps010_peer_half_seed2`

See apps/cifar100_ju_k_heads4_r18_tiny_eps010_peer_half_seed2.yml for every setting and why.


## Before you start

* Accelerator **GPU T4 x2**, Internet **On**
* Add the CIFAR-100 dataset as an input
* **Save Version -> Save & Run All (Commit)**, not the interactive run.
  An interactive session ends when the browser closes.

The checks at the top run on the CPU and stop the session in about two
minutes if anything is wrong, before a card is touched. Note that the
branch suite mimics the training loop rather than running it, so for
these branches the smoke step below is what actually exercises the new
code. Do not skip it.

If the session times out partway, carry it forward rather than starting
over. Every epoch writes a checkpoint, so at most one is lost, and the
learning-rate schedule is restored with it - resuming does not restart
the cosine.

1. Save Version on the timed-out session, so its output persists.
2. Make a fresh copy of this notebook.
3. Add data -> Notebook Output -> pick that session.
4. Set `RESUME_FROM` to the directory that *contains* `cifar100_<branch>`,
   then Save & Run All.

That directory is in one of two places depending on how the session
ended, because the copy at the bottom of this notebook only runs if the
session got that far:

    finished the last cell   /kaggle/input/<slug>/logs
    killed mid-training      /kaggle/input/<slug>/new-pruning/logs

If in doubt, run `!find /kaggle/input -name latest_checkpoint.pt` in a
scratch cell and take the parent of the parent. A correct path prints
`restored cifar100_<branch>` and then `Loaded checkpoint ... at epoch N`;
if you see `starting from scratch` instead, the path is wrong and the run
will silently begin again at epoch 1.

## When it finishes

Send back the final table. Pasting the output of the last cell is enough.

In [ ]:
# Fixed for this notebook. Notebook 125.
BRANCHES = ['jt_k_heads4_r18_tiny_eps010_peer_half', 'ju_k_heads4_r18_tiny_eps010_peer_half_seed2']

SMOKE_FIRST = True

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

# To carry a timed-out session forward, attach its output and name the
# logs directory. Leave empty to start fresh.
RESUME_FROM = ''

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
for i in range(n_gpu):
    print('  {}: {}'.format(i, torch.cuda.get_device_properties(i).name))

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)
print('code at', subprocess.run(
    ['git', 'log', '-1', '--format=%h %s'], capture_output=True,
    text=True).stdout.strip())

available = sorted(
    name[len('cifar100_'):-len('.yml')]
    for name in os.listdir('apps')
    if name.startswith('cifar100_') and name.endswith('.yml'))
print('\nbranches in apps/:')
for name in available:
    print('   ', name)

missing = [b for b in BRANCHES if b not in available]
if missing:
    raise SystemExit('no config for {}'.format(missing))
if n_gpu == 0:
    raise SystemExit('No GPU in this session. Set Accelerator to '
                     'GPU T4 x2 in the notebook settings and run again.')
if n_gpu < len(BRANCHES):
    print('\n{} branches, {} gpu(s): they will run in sequence.'.format(
        len(BRANCHES), n_gpu))

In [ ]:
# What the chosen branches actually differ in, read off the configs
# rather than from the table above, which can drift.
AXES = ('kd_loss', 'cost_source', 'feature_kd', 'feature_align',
        'feature_layers', 'feature_weight', 'tier_weights',
        'horizontal_kd', 'horizontal_where', 'horizontal_loss',
        'horizontal_weight', 'weight_schedule')

settings = {}
for branch in BRANCHES:
    found = {}
    with open('apps/cifar100_{}.yml'.format(branch)) as handle:
        for line in handle:
            key = line.split(':')[0].strip()
            if key in AXES:
                found[key] = line.split(':', 1)[1].strip()
    settings[branch] = found

print('{:20}'.format('') + ''.join(
    '{:>26}'.format(b) for b in BRANCHES))
for axis in AXES:
    values = [settings[b].get(axis, '-') for b in BRANCHES]
    if all(v == '-' for v in values):
        continue
    print('{:20}'.format(axis) + ''.join(
        '{:>26}'.format(v) for v in values))

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

## Checks, before a card is touched

Seconds on the CPU. Between them these suites have caught a Sinkhorn solved too loosely to have a correct gradient, an alpha-divergence that destroyed the weights in three steps, a calibration that reset the batch norm statistics and never refilled them, and a feature cost normalized so that its own gradient vanished. Every one of those was silent.

The last suite builds each branch in this notebook and runs two training steps of the real loop, profiling included. Four feature branches once reached Kaggle, passed every loss check, and died in the profiler on the first forward, because nothing local had ever called it.

In [ ]:
# Driven by BRANCHES, not by a list written beside it. The branch suite
# used to name its two branches literally, which quietly checked the wrong
# pair for anyone who changed BRANCHES and nothing else.
suites = ['tests/test_loss_ops.py', 'tests/test_bn_calibration.py',
          'tests/test_kd_variants.py',
          'tests/test_channel_reorder.py',
          'tests/test_published_methods.py',
          'tests/test_freeze.py',
          'tests/test_head_groups.py',
          'tests/test_affine_knots.py',
          'tests/test_no_kd.py',
          'tests/test_pre_relu.py',
          'tests/test_dynas.py',
          'tests/test_feature_start.py',
          'tests/test_amp.py',
          'tests/test_wkd.py',
          'tests/test_stratified.py',
          'tests/test_lora.py',
          'tests/test_sam.py']
for suite in suites:
    done = subprocess.run([sys.executable, suite], capture_output=True,
                          text=True)
    print('{:34} {}'.format(
        suite, (done.stdout.strip().splitlines() or ['no output'])[-1]))
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-2000:])
        raise SystemExit('{} failed'.format(suite))

done = subprocess.run(
    [sys.executable, 'tests/test_all_branches.py'] + BRANCHES,
    capture_output=True, text=True)
print()
print(done.stdout.strip()[-2000:])
if done.returncode != 0:
    print(done.stderr[-2000:])
    raise SystemExit('a branch in {} does not build'.format(BRANCHES))

In [ ]:
if not RESUME_FROM:
    found = set()
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        if ('latest_checkpoint.pt' in files and
                os.path.basename(root)[len('cifar100_'):] in BRANCHES):
            found.add(os.path.dirname(root))
    if len(found) > 1:
        raise SystemExit('checkpoints in more than one input, set '
                         'RESUME_FROM to one of: {}'.format(sorted(found)))
    if found:
        RESUME_FROM = found.pop()
        print('RESUME_FROM was empty; found checkpoints in', RESUME_FROM)
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
            print('restored', name)
else:
    print('starting from scratch')
for branch in BRANCHES:
    ckpt = os.path.join('logs', 'cifar100_' + branch,
                        'latest_checkpoint.pt')
    print(branch, 'resumes from its checkpoint' if os.path.exists(ckpt)
          else 'starts at epoch 1')

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs, quiet=True):
    """one job per card, output interleaved and tagged"""
    lines = queue.Queue()
    procs = {}
    failing = set()

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1))
        # Once a branch starts printing a traceback, stop filtering
        # it. The body is indented, so the rule below would keep the
        # exception and throw away where it came from.
        if 'Traceback (most recent call last)' in line:
            failing.add(label)
        if quiet and label not in failing and (
                line.startswith(('  ', ')', 'Model(', 'Total', 'Item'))
                or not line.strip()):
            continue
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit('smoke failed for {}'.format(failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    results.clear()
    print('\nsmoke ok')

In [ ]:
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results, against SlimOT at the same seed

The "SlimOT" column is SlimOT as in the paper at the same seed; US-Net and AlphaNet are at the same seed too.

In [ ]:
# SlimOT as in the paper (blur 0.2, Peer Transport 1.0), same seed:
# EQ (1995), FP (2026), FR (2006)
SLIMOT = {
    'jt_k_heads4_r18_tiny_eps010_peer_half': [54.93, 56.03, 56.56, 57.92, 58.43, 59.18, 59.72, 59.8, 59.47, 60.05, 60.41, 60.9, 61.18, 60.9, 61.23, 61.62],
    'ju_k_heads4_r18_tiny_eps010_peer_half_seed2': [54.77, 55.85, 56.91, 57.66, 58.09, 58.6, 58.73, 59.11, 59.41, 59.62, 59.8, 60.18, 60.19, 60.65, 60.65, 60.79],
    'jv_k_heads4_r18_tiny_eps010_peer_half_seed3': [54.61, 55.47, 56.61, 57.09, 57.98, 58.81, 59.41, 59.49, 59.69, 59.95, 60.12, 60.4, 60.38, 60.39, 60.78, 61.2],
}
# US-Net (ES, FO, FQ) and AlphaNet (GE, GF, GG), same three seeds
USNET = {
    'jt_k_heads4_r18_tiny_eps010_peer_half': [54.98, 55.5, 55.96, 56.43, 57.48, 57.26, 57.69, 58.27, 58.79, 59.18, 59.41, 59.39, 59.72, 59.54, 59.98, 60.46],
    'ju_k_heads4_r18_tiny_eps010_peer_half_seed2': [54.79, 55.89, 56.34, 57.18, 57.41, 57.95, 58.4, 58.92, 59.38, 59.61, 59.41, 59.81, 60.45, 60.38, 60.59, 61.08],
    'jv_k_heads4_r18_tiny_eps010_peer_half_seed3': [55.25, 55.75, 56.64, 57.05, 57.82, 58.79, 59.0, 59.27, 59.86, 59.93, 59.93, 60.67, 60.9, 60.89, 61.02, 61.23],
}
ALPHANET = {
    'jt_k_heads4_r18_tiny_eps010_peer_half': [55.62, 56.03, 56.96, 57.42, 58.16, 58.99, 59.83, 59.54, 60.18, 60.5, 60.67, 60.44, 60.8, 60.92, 60.75, 60.86],
    'ju_k_heads4_r18_tiny_eps010_peer_half_seed2': [54.75, 55.7, 56.55, 57.31, 58.4, 59.09, 59.38, 59.75, 59.9, 59.74, 60.04, 60.29, 60.53, 60.61, 60.87, 60.98],
    'jv_k_heads4_r18_tiny_eps010_peer_half_seed3': [55.32, 56.16, 56.91, 57.21, 58.29, 58.89, 59.58, 59.33, 59.46, 59.79, 59.63, 60.0, 60.58, 60.61, 60.53, 60.72],
}
WIDTHS = [0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]


def mean(values):
    return sum(values) / len(values)


for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        print(branch, 'no results')
        continue
    ours = {round(w, 2): 100.0 * (1.0 - v[1]) for w, v in table.items()}
    print(branch)
    print('{:>7}{:>9}{:>9}{:>9}{:>9}{:>9}'.format(
        'width', 'JM set', 'SlimOT', 'vs', 'US-Net', 'AlphaNet'))
    for i, width in enumerate(WIDTHS):
        if width not in ours:
            continue
        print('{:>7.2f}{:>9.2f}{:>9.2f}{:>+9.2f}{:>9.2f}{:>9.2f}'.format(
            width, ours[width], SLIMOT[branch][i], ours[width] - SLIMOT[branch][i],
            USNET[branch][i], ALPHANET[branch][i]))
    accuracies = [ours[w] for w in WIDTHS if w in ours]
    above = sum(ours[w] > SLIMOT[branch][i] for i, w in enumerate(WIDTHS) if w in ours)
    print('mean {:.2f}   SlimOT {:.2f} ({:+.2f}, above at {}/{} widths)'
          '   US-Net {:.2f}   AlphaNet {:.2f}\n'.format(
              mean(accuracies), mean(SLIMOT[branch]),
              mean(accuracies) - mean(SLIMOT[branch]), above, len(accuracies),
              mean(USNET[branch]), mean(ALPHANET[branch])))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'cifar100_' + branch),
                        dirs_exist_ok=True)
print('checkpoints copied to', out)